In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0] else (False, sys.argv[1])

# START

In [2]:
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import connectionscanmodel, stepmodel
from quetzal.io import excel
import numpy as np
import pandas as pd
import geopandas as gpd

import shapely

import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'models/{}/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

In [5]:
if var['run_mode']['mode'] == 'timetables':
    print('timetables scenario')
    end_of_notebook

In [6]:
NTLEG_PENALTY = var[('pathfinders_general', 'ntleg_penalty')]
TRANSFER_DURATION = var[('pathfinders_general', 'transfer_duration')]
WALK_SPEED = var[('pathfinders_general', 'walk_speed')]
BIKE_SPEED = var[('pathfinders_general', 'bike_speed')]
TRANSFER_PENALTY = var[('pathfinders_general', 'transfer_penalty')]

WALK_TIME_FACTOR = var[('mode_penalty', 'walk_time_factor')]
BIKE_TIME_FACTOR = var[('mode_penalty', 'bike_time_factor')]
BOARDING_TIMES = var['boarding_time'].to_dict()

FLAT_FARES = var['fare'].to_dict()
KM_FARES = var['fare_km'].to_dict()

VALUE_OF_TIME_SEC = var[('value_of_time', 'value_of_time_h')]/60

INTRA_AREA_TO_DISTANCE_FACTOR = var[('step_distribution', 'intra_area_to_distance_factor')]

# READ

In [7]:
sm = stepmodel.read_zippedpickles(mpath + 'zones')

zones: 100%|██████████| 13/13 [00:00<00:00, 41.38it/s]


# NODE CLUSTERING

In [8]:
sm.preparation_clusterize_nodes(
     distance_threshold=200
)

In [9]:
sm._add_type_prefixes(
        prefixes={'nodes': 'node_'}
    )

can not add prefixes on table:  footpaths


# Walkmodels

In [10]:
sm.road_links['walk_speed'] = WALK_SPEED # km / h
sm.road_links['walk_time'] = sm.road_links.length / (sm.road_links['walk_speed'] / 3.6) * WALK_TIME_FACTOR

In [11]:
sm.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    short_leg_speed=WALK_SPEED/WALK_TIME_FACTOR,
    long_leg_speed=WALK_SPEED/WALK_TIME_FACTOR
)

In [12]:
sm._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

# PATHFINDER - PT ONLY

In [13]:
# Access times
sm.links['time'] += sm.links['route_type'].map(BOARDING_TIMES).fillna(0)

In [14]:
# add fare in time for pathfinder to find most relevant paths
sm.links['fare_km'] = sm.links.length * sm.links.route_type.apply(lambda x: KM_FARES[x]) / 1000
sm.links['true_time'] = sm.links['time'].copy()
sm.links['time'] = sm.links['time'] + (sm.links['fare_km'] / VALUE_OF_TIME_SEC)

In [15]:
# pruning algorithm
sm.step_pt_pathfinder(
    broken_modes=True, broken_routes=True,
    walk_on_road=True
)

# remove intrazonal (handled differently)
sm.pt_los = sm.pt_los[sm.pt_los['origin']!=sm.pt_los['destination']]
sm.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
18 + 25 = 43 >= 0.5 * 68 keeping od_set
59 + 25 = 84 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
15 + 16 = 31 < 0.5 * 67 splitting od_set
0 + 1 = 1 >= 0.5 * 1 keeping od_set
0 + 1 = 1 >= 0.5 * 1 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
6 + 5 = 11 < 0.5 * 68 splitting od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
3 + 1 = 4 < 0.5 * 19 splitting od_set
1 + 1 = 2 >= 0.5 * 2 keeping od_set
18 + 12 = 30 < 0.5 * 67 splitting od_set
2 + 1 = 3 < 0.5 * 67 splitting od_set
0 + 1 = 1 >= 0.5 * 1 keeping od_set
2 + 1 = 3 < 0.5 * 9 splitting od_set
3 + 1 = 4 >= 0.5 * 8 keeping od_set
18 + 2 = 20 >= 0.5 * 23 keeping od_set
3 + 5 = 8 >= 0.5 * 10 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_

  0%|          | 0/98 [00:00<?, ?it/s]

15 sources 67 targets direct search
53 sources 16 targets transposed search


route_id {'82'}:   5%|▌         | 5/98 [00:00<00:10,  8.83it/s] 

7 sources 1 targets transposed search
67 sources 1 targets transposed search
1 sources 1 targets direct search
1 sources 3 targets direct search


route_id {'F43'}:   7%|▋         | 7/98 [00:00<00:10,  8.97it/s]

... (78 outputs left out)

68 sources 68 targets direct search


route_type {'tramway', 'bus'}: 100%|██████████| 7/7 [00:04<00:00,  1.70it/s]


path_analysis


path_analysis: 100%|██████████| 32542/32542 [00:01<00:00, 19523.82it/s]


In [16]:
hierarchy = ['rail', 'tramway', 'bus', 'schoolbus', 'ferry', 'walk']
sm.analysis_pt_route_type(hierarchy)

In [17]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm.links.route_type.unique():
    links = sm.links[sm.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm.pt_los[k + '_link_path'] = sm.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm.pt_los[k + '_in_vehicle_length'] = sm.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm.pt_los['fare'] = 0
for m in links_mode.keys():
    sm.pt_los['fare'] += (sm.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

# PATHFINDER - PT + BIKE

In [18]:
sm_velo = sm.copy()

In [19]:
sm_velo.road_links['walk_speed'] = BIKE_SPEED
sm_velo.road_links['walk_time'] = sm_velo.road_links.length / (sm_velo.road_links['walk_speed'] / 3.6) * BIKE_TIME_FACTOR

In [20]:
### TODO: filtrer sur modes velo compatibles

# pruning algorithm
sm_velo.step_pt_pathfinder(
    broken_modes=True, broken_routes=False,
    walk_on_road=True
)

# remove intrazonal (handled differently)
sm_velo.pt_los = sm_velo.pt_los[sm_velo.pt_los['origin']!=sm_velo.pt_los['destination']]
sm_velo.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
68 + 0 = 68 >= 0.5 * 68 keeping od_set
53 + 30 = 83 >= 0.5 * 68 keeping od_set
23 + 26 = 49 >= 0.5 * 68 keeping od_set
28 + 22 = 50 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
54 + 44 = 98 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set


  0%|          | 0/7 [00:00<?, ?it/s]

68 sources 68 targets direct search


route_type {'tramway', 'rail'}:  29%|██▊       | 2/7 [00:01<00:03,  1.45it/s]       

68 sources 68 targets direct search
68 sources 68 targets direct search


route_type {'tramway'}:  57%|█████▋    | 4/7 [00:02<00:02,  1.42it/s]        

... (2 outputs left out)

68 sources 68 targets direct search
68 sources 68 targets direct search


route_type {'tramway', 'bus'}: 100%|██████████| 7/7 [00:04<00:00,  1.47it/s]


path_analysis


path_analysis: 100%|██████████| 13967/13967 [00:01<00:00, 10072.75it/s]


In [21]:
hierarchy = ['rail', 'tramway', 'bus', 'schoolbus', 'ferry', 'bike']
sm_velo.analysis_pt_route_type(hierarchy)

In [22]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm_velo.links.route_type.unique():
    links = sm_velo.links[sm_velo.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm_velo.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm_velo.pt_los[k + '_link_path'] = sm_velo.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm_velo.pt_los[k + '_in_vehicle_length'] = sm_velo.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm_velo.pt_los['fare'] = 0
for m in links_mode.keys():
    sm_velo.pt_los['fare'] += (sm_velo.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm_velo.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

# Correct times

In [23]:
# TC
sm.links['pathfinder_time'] = sm.links['time']
sm.links['time'] = sm.links['true_time']
sm.road_links['walk_time'] *= 1 / WALK_TIME_FACTOR
sm.zone_to_road['time'] *= 1 / WALK_TIME_FACTOR
sm.analysis_pt_time(boarding_time=0, walk_on_road=True)

# Vélo
sm_velo.links['pathfinder_time'] = sm_velo.links['time']
sm_velo.links['time'] = sm_velo.links['true_time']
sm_velo.road_links['walk_time'] *= 1 / WALK_TIME_FACTOR
sm_velo.zone_to_road['time'] *= 1 / WALK_TIME_FACTOR
sm_velo.analysis_pt_time(boarding_time=0, walk_on_road=True)

In [24]:
sm.pt_los = pd.concat([sm.pt_los, sm_velo.pt_los])

# ROAD PATHFINDER | car

In [25]:
# empty volumes
sm.volumes = pd.DataFrame(index=pd.MultiIndex.from_product([sm.zones.index]*2))
sm.volumes.index.names = ['origin', 'destination']
sm.volumes.reset_index(inplace=True)

In [26]:
sm.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm.car_los = sm.car_los[sm.car_los['origin']!=sm.car_los['destination']]
sm.car_los['pathfinder_session'] = 'car_aon'

In [27]:
sm.analysis_car_los()
sm.analysis_car_length()

# ROAD PATHFINDER | bike

In [28]:
sm_road_bike = sm.copy()
### No bikes on motorways
# TODO: remplacer ça par un attribut "bike" à False dans le réseau routier et 
# mettre cette préparation dans le script networks (integrity_fix_road_network sert à modifier l’attribut bike),
# et utiliser ce réseau également pour les footpaths précédents (vélo+TC)
sm_road_bike.road_links = sm_road_bike.road_links.loc[~sm_road_bike.road_links['highway'].isin(['motorway', 'motorway_link', 'trunk', 'trunk_link'])]
sm_road_bike.road_nodes = sm_road_bike.road_nodes.loc[sm_road_bike.road_nodes.index.isin(sm_road_bike.road_links['a'].values.tolist()) | sm_road_bike.road_nodes.index.isin(sm_road_bike.road_links['b'].values.tolist())]

sm_road_bike.integrity_fix_road_network()

sm_road_bike.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    short_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR,
    long_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR
)
sm_road_bike._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

sm_road_bike.road_links['speed'] = BIKE_SPEED
sm_road_bike.road_links['time'] = sm_road_bike.road_links.length / (sm_road_bike.road_links['speed'] / 3.6) * BIKE_TIME_FACTOR

direct: 53444it [00:01, 42982.83it/s]
reversed: 53444it [00:01, 46331.32it/s]
53330it [00:01, 40408.28it/s]


In [29]:
sm_road_bike.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm_road_bike.car_los = sm_road_bike.car_los[sm_road_bike.car_los['origin']!=sm_road_bike.car_los['destination']]
sm_road_bike.car_los['pathfinder_session'] = 'car_aon'

In [30]:
sm_road_bike.analysis_car_los()
sm_road_bike.analysis_car_length()

# INTRAZONAL

In [31]:
sm.zone_length = (sm.zones.area ** 0.5) * INTRA_AREA_TO_DISTANCE_FACTOR

In [32]:
def set_intra_los(mode, speed_kmh, km_fare, base_fare=0, time_penalty=0):
    intra_los = sm.volumes[sm.volumes['origin']==sm.volumes['destination']].copy()

    intra_los['route_type'] = mode
    if mode != 'walk':
        intra_los['route_types'] = [(mode,)]*len(intra_los)
    else:
        intra_los['route_types'] = [tuple()]*len(intra_los)

    intra_los['length'] = intra_los['origin'].apply(lambda x: sm.zone_length[x])
    intra_los['gtime'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] / (speed_kmh / 3.6)
    )
    intra_los['gtime'] += time_penalty
    intra_los['time'] = intra_los['gtime']
    intra_los['fare'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] * km_fare / 1000 + base_fare
    )
    intra_los['pathfinder_session'] = 'intrazonal'
    intra_los['ntransfers'] = 0
    intra_los['path'] = intra_los['origin'].apply(lambda x: tuple([x]))

    for col in ['node_path', 'link_path', 'boardings', 'alightings',
        'footpaths', 'ntlegs', 'boarding_links', 'alighting_links', 'transfers']:
            intra_los[col] = [tuple()]*len(intra_los)

    return intra_los

sm.intra_los = pd.concat(
    [
        set_intra_los('car', 40, KM_FARES['car'], base_fare=0),
        set_intra_los('walk', WALK_SPEED, 0)
    ]
)

sm.intra_los.reset_index(drop=True, inplace=True)

# CONCAT

In [33]:
# CAR
sm.car_los['penalties'] = 0 ### PARKING_PENALTY + CAR_ACCESS_PENALTY
sm.car_los['ntransfers'] = 0
sm.car_los['fare'] = sm.car_los['length'] * KM_FARES['car'] / 1000
sm.car_los['gtime'] *= 1 ### CAR_TIME_FACTOR
sm.car_los['gtime'] += sm.car_los['penalties']
sm.car_los['route_types'] = [('car',)] * len(sm.car_los)
sm.car_los['route_type'] = 'car'
sm.car_los['time'] = sm.car_los['gtime']
sm.car_los['gtime'] += sm.car_los['in_vehicle_length'] / 1000 * KM_FARES['car'] / VALUE_OF_TIME_SEC

# BIKE
sm_road_bike.car_los['penalties'] = 0 ### PARKING_PENALTY + CAR_ACCESS_PENALTY
sm_road_bike.car_los['ntransfers'] = 0
sm_road_bike.car_los['fare'] = 0
sm_road_bike.car_los['gtime'] *= 1 ### CAR_TIME_FACTOR
sm_road_bike.car_los['gtime'] += sm.car_los['penalties']
sm_road_bike.car_los['route_types'] = [('bike',)] * len(sm.car_los)
sm_road_bike.car_los['route_type'] = 'bike'
sm_road_bike.car_los['time'] = sm.car_los['gtime']

sm.los = pd.concat([sm.car_los, sm_road_bike.car_los, sm.pt_los, sm.intra_los]).reset_index(drop=True)

# EXPORT

In [ ]:
### TODO: enlever schoolbus du réseau TC

print(sm.los['route_types'].unique())
# TODO Louison: pourquoi on a pas de route_types bike + autre-chose ?
# print(sm.los['route_type'].unique())
sm.to_zippedpickles(
    mpath + r'pathfinders',
    omitted_attributes=['pt_los', 'car_los', 'intra_los']
)

[('car',) ('bike',) ('bus',) ('bus', 'rail') ('bus', 'rail', 'tramway')
 ('bus', 'tramway') ('rail',) ('rail', 'tramway') () ('tramway',)]


los: 100%|██████████| 24/24 [00:04<00:00,  5.23it/s]            
